# Veridelta Quickstart

Veridelta is a high-performance semantic diffing engine built on Polars. Standard binary equality checks often fail in production pipelines due to trivial transformation artifacts like floating-point noise, string formatting variance, or schema evolution. 

Veridelta enables data engineers to define **logical equality** through declarative configuration, ensuring resilient data validation at scale.

## 1. The Challenge: Physical vs. Logical Equality

In data migration or ETL workflows, standard equality checks fail when introducing minor, mathematically insignificant variances (e.g., rounding errors or inconsistent string padding). 

The following example demonstrates logically identical data failing a strict physical comparison.

In [1]:
import polars as pl

# Source System (Legacy)
src_df = pl.DataFrame(
    {"id": [1, 2, 3], "amount": [100.0, 250.50, 300.0], "status": ["active", "inactive", "pending"]}
)

# Target System (Modernized) - Introduces jitter and padding
tgt_df = pl.DataFrame(
    {
        "id": [1, 2, 3],
        "amount": [100.04, 250.49, 300.0],
        "status": [" ACTIVE ", "INACTIVE", " PENDING "],
    }
)

is_physically_equal = src_df.equals(tgt_df)
print(f"Strict Physical Equality: {is_physically_equal}")
# Output: Strict Physical Equality: False

Strict Physical Equality: False


## 2. Defining Semantic Boundaries

Veridelta resolves physical variance by establishing semantic boundaries. The `DiffConfig` object declaratively defines acceptable tolerances and normalization rules. By decoupling the configuration from the execution engine, Veridelta seamlessly supports both in-memory and out-of-core workflows.

In [2]:
from veridelta import ColumnRule, DiffConfig, DiffEngine

# Define the semantic comparison boundaries
config = DiffConfig(
    primary_keys=["id"],
    # Global Policies: Applied to all applicable columns unless overridden
    default_absolute_tolerance=0.05,
    default_whitespace_mode="both",
    # Granular Overrides: Column-specific normalization
    rules=[ColumnRule(column_names=["status"], case_insensitive=True)],
)

# Execute the comparison using the aligned DataFrames
engine = DiffEngine(config, src_df, tgt_df)
summary = engine.run()

print(f"Match Status: {summary.is_match}")
print(f"Changed Rows: {summary.changed_count}")
# Output: Match Status: True
# Output: Changed Rows: 0

Match Status: True
Changed Rows: 0


## 3. Enterprise Ingestion & Semantic Drift Simulation

For production workloads, Veridelta provides a `DataIngestor` to handle remote I/O, header normalization, and schema alignment natively. 

The following example fetches a remote Parquet dataset (January 2026 NYC Taxi Records), programmatically injects transformation noise to simulate downstream drift, and executes a multi-threaded semantic comparison.

In [3]:
from veridelta import DataIngestor, SourceConfig

DATA_URL = "https://raw.githubusercontent.com/Veridelta/veridelta/main/docs/assets/data/sample_taxi_data.parquet"

# 1. Define configuration for the target schema
config = DiffConfig(
    primary_keys=["id"],
    default_absolute_tolerance=0.01,
    default_whitespace_mode="both",
    rules=[ColumnRule(column_names=["store_and_fwd_flag"], case_insensitive=True)],
)

# 2. Ingest remote artifacts
source_io = SourceConfig(path=DATA_URL, format="parquet")
ingestor = DataIngestor(config, source_io, source_io)

# In a real workflow, target_io would point to the downstream system
src_df, _ = ingestor.get_dataframes()

# 3. Simulate downstream ETL drift (floating-point noise & padding)
tgt_df = src_df.with_columns(
    [
        (pl.col("fare_amount") + 0.002).alias("fare_amount"),
        (pl.col("store_and_fwd_flag") + " ").alias("store_and_fwd_flag"),
    ]
)

# 4. Execute the semantic diff engine
engine = DiffEngine(config, src_df, tgt_df)
summary = engine.run()

# 5. Review Execution Metrics
print(f"{' VERIDELTA EXECUTION SUMMARY ':=^50}")
print(f"Match Status:     {'SUCCESS' if summary.is_match else 'FAILED'}")
print(f"Total Rows:       {summary.total_rows_source:,}")
print(f"Semantic Diffs:   {summary.changed_count:,}")
print("=" * 50)

OSError: object-store error: Object at location  not found: Error performing HEAD https://raw.githubusercontent.com/Veridelta/veridelta/main/docs/assets/data/sample_taxi_data.parquet in 78.568232ms - Server returned non-2xx status code: 404 Not Found:  (path: https://raw.githubusercontent.com/Veridelta/veridelta/main/docs/assets/data/sample_taxi_data.parquet)

## Architectural Highlights

Veridelta is engineered for integration into high-throughput data pipelines and CI/CD validation gates.

* **Zero-Import Configuration:** Semantic modes utilize literal strings, enabling configuration via YAML/JSON without requiring Python imports.
* **Strict Type Validation:** Built on Pydantic V2, ensuring configuration logic is strictly validated before expensive I/O operations begin.
* **Hardware-Accelerated Execution:** Direct integration with Polars' Rust backend ensures that normalization and row-level diffing scales linearly with available CPU cores.